# Offline RAG Knowledge Base Audit



In [1]:
from pathlib import Path
import pandas as pd
import sqlite3
import json

current = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:
    if (
        (candidate / "research").is_dir()
        and (candidate / "pubspec.yaml").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

RESEARCH_DIR = PROJECT_ROOT / "research"

RAG_DIR = RESEARCH_DIR / "rag"

SOURCE_DIR = (
    RAG_DIR / "source_documents"
)

PROCESSED_DIR = (
    RAG_DIR / "processed"
)

DATABASE_DIR = (
    RAG_DIR / "database"
)

EVALUATION_DIR = (
    RAG_DIR / "evaluation"
)

for directory in [
    SOURCE_DIR,
    PROCESSED_DIR,
    DATABASE_DIR,
    EVALUATION_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("OFFLINE RAG ENVIRONMENT")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("RAG directory:", RAG_DIR)
print("Source directory:", SOURCE_DIR)
print("Database directory:", DATABASE_DIR)

OFFLINE RAG ENVIRONMENT
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
RAG directory: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag
Source directory: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/source_documents
Database directory: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database


In [2]:
RAG_CATEGORIES = [
    "general_first_aid",
    "hypothermia",
    "heat_illness",
    "heavy_rain",
    "strong_wind",
    "water_safety",
    "remote_emergency"
]

print("OFFLINE KNOWLEDGE CATEGORIES")
print("=" * 60)

for index, category in enumerate(
    RAG_CATEGORIES,
    start=1
):
    print(
        f"{index}. {category}"
    )

print()
print(
    "Total categories:",
    len(RAG_CATEGORIES)
)

OFFLINE KNOWLEDGE CATEGORIES
1. general_first_aid
2. hypothermia
3. heat_illness
4. heavy_rain
5. strong_wind
6. water_safety
7. remote_emergency

Total categories: 7


In [3]:
CATALOG_FILE = (
    RAG_DIR / "source_catalog.csv"
)

print("SOURCE CATALOG")
print("=" * 60)

print(
    "Catalog exists:",
    CATALOG_FILE.exists()
)

if not CATALOG_FILE.exists():
    raise FileNotFoundError(
        f"Missing source catalog: {CATALOG_FILE}"
    )

catalog = pd.read_csv(
    CATALOG_FILE
)

print(
    "Catalog shape:",
    catalog.shape
)

print(
    "Columns:"
)

for column in catalog.columns:
    print(" -", column)

display(
    catalog.head()
)

SOURCE CATALOG
Catalog exists: True
Catalog shape: (17, 15)
Columns:
 - source_id
 - title
 - organization
 - categories
 - source_url
 - official_domain
 - publication_date
 - retrieved_date
 - language
 - file_type
 - local_path
 - download_status
 - authoritative
 - primary_source
 - notes


,source_id,title,organization,categories,source_url,official_domain,publication_date,retrieved_date,language,file_type,local_path,download_status,authoritative,primary_source,notes
0,SRC_001,"IFRC International First Aid, Resuscitation an...",IFRC,general_first_aid;hypothermia;heat_illness;wat...,https://www.ifrc.org/sites/default/files/2026-...,www.ifrc.org,2026-03-18,2026-10-18,en,pdf,source_documents/general_first_aid/ifrc_first_...,downloaded,True,True,2025 edition; official IFRC listing dates rele...
1,SRC_002,Preventing Hypothermia,Centers for Disease Control and Prevention (CDC),hypothermia,https://www.cdc.gov/winter-weather/prevention/...,www.cdc.gov,2024-05-15,2026-10-18,en,html,NaN,url_only,True,True,US context; preserve distinctions between hypo...
2,SRC_003,Heat-related Illnesses,Centers for Disease Control and Prevention (CD...,heat_illness,https://www.cdc.gov/niosh/heat-stress/about/il...,www.cdc.gov,NaN,2026-10-18,en,html,NaN,url_only,True,True,Worker-oriented guidance; exertional exposure ...
3,SRC_004,"About Water Treatment Options When Hiking, Cam...",Centers for Disease Control and Prevention (CDC),water_safety,https://www.cdc.gov/drinking-water/prevention/...,www.cdc.gov,NaN,2026-10-18,en,html,NaN,url_only,True,True,Product instructions and contaminant-specific ...
4,SRC_005,During a Flood,NOAA National Weather Service (NWS),heavy_rain;water_safety,https://www.weather.gov/safety/flood-during,www.weather.gov,2018-03-12,2026-10-18,en,html,source_documents/heavy_rain/nws_during_a_flood...,downloaded,True,True,US emergency numbers and warning systems are n...


In [4]:
REQUIRED_CATALOG_COLUMNS = [
    "source_id",
    "title",
    "organization",
    "categories",
    "source_url",
    "official_domain",
    "publication_date",
    "retrieved_date",
    "language",
    "file_type",
    "local_path",
    "download_status",
    "authoritative",
    "primary_source",
    "notes"
]

missing_columns = [
    column
    for column in REQUIRED_CATALOG_COLUMNS
    if column not in catalog.columns
]

print("CATALOG SCHEMA VALIDATION")
print("=" * 60)

print(
    "Missing required columns:",
    missing_columns
)

print(
    "Duplicate source IDs:",
    catalog["source_id"]
    .duplicated()
    .sum()
)

critical_columns = [
    "source_id",
    "title",
    "organization",
    "categories",
    "source_url"
]

print("\nMissing critical metadata:")

print(
    catalog[
        critical_columns
    ]
    .isna()
    .sum()
)

print(
    "\nSchema valid:",
    len(missing_columns) == 0
)

CATALOG SCHEMA VALIDATION
Missing required columns: []
Duplicate source IDs: 0

Missing critical metadata:
source_id       0
title           0
organization    0
categories      0
source_url      0
dtype: int64

Schema valid: True


In [5]:
print("DOWNLOAD STATUS")
print("=" * 60)

print(
    catalog["download_status"]
    .value_counts(
        dropna=False
    )
)

print()
print(
    "Total catalog sources:",
    len(catalog)
)

DOWNLOAD STATUS
download_status
downloaded    11
url_only       6
Name: count, dtype: int64

Total catalog sources: 17


In [6]:
def resolve_local_path(local_path):

    if pd.isna(local_path):
        return None

    value = str(
        local_path
    ).strip()

    if not value:
        return None

    path = Path(value)

    if path.is_absolute():
        return path

    # Catalog paths are expected to be relative
    # to research/rag/
    return RAG_DIR / path


catalog[
    "resolved_local_path"
] = catalog[
    "local_path"
].apply(
    resolve_local_path
)

catalog[
    "local_file_exists"
] = catalog[
    "resolved_local_path"
].apply(
    lambda path:
        path is not None
        and path.is_file()
)

catalog[
    "local_file_size_bytes"
] = catalog[
    "resolved_local_path"
].apply(
    lambda path:
        path.stat().st_size
        if (
            path is not None
            and path.is_file()
        )
        else 0
)

print("LOCAL FILE VALIDATION")
print("=" * 60)

print(
    "Catalog sources:",
    len(catalog)
)

print(
    "Local files found:",
    int(
        catalog[
            "local_file_exists"
        ].sum()
    )
)

print(
    "Missing local files:",
    int(
        (
            ~catalog[
                "local_file_exists"
            ]
        ).sum()
    )
)

print(
    "Zero-byte local files:",
    int(
        (
            catalog[
                "local_file_exists"
            ]
            &
            (
                catalog[
                    "local_file_size_bytes"
                ] == 0
            )
        ).sum()
    )
)

LOCAL FILE VALIDATION
Catalog sources: 17
Local files found: 11
Missing local files: 6
Zero-byte local files: 0


In [7]:
offline_catalog = (
    catalog[
        catalog[
            "local_file_exists"
        ]
        &
        (
            catalog[
                "local_file_size_bytes"
            ] > 0
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

print("OFFLINE-ELIGIBLE KNOWLEDGE SOURCES")
print("=" * 60)

print(
    "Offline sources:",
    len(offline_catalog)
)

print(
    "Excluded URL-only/missing sources:",
    len(catalog)
    - len(offline_catalog)
)

display(
    offline_catalog[
        [
            "source_id",
            "title",
            "organization",
            "categories",
            "file_type",
            "local_file_size_bytes"
        ]
    ]
)

OFFLINE-ELIGIBLE KNOWLEDGE SOURCES
Offline sources: 11
Excluded URL-only/missing sources: 6


,source_id,title,organization,categories,file_type,local_file_size_bytes
0,SRC_001,"IFRC International First Aid, Resuscitation an...",IFRC,general_first_aid;hypothermia;heat_illness;wat...,pdf,6261070
1,SRC_005,During a Flood,NOAA National Weather Service (NWS),heavy_rain;water_safety,html,26657
2,SRC_006,Lightning Safety,NOAA National Weather Service (NWS),heavy_rain,html,26044
3,SRC_007,During a High Wind Event,NOAA National Weather Service (NWS),strong_wind,html,27157
4,SRC_008,Before a High Wind Event,NOAA National Weather Service (NWS),strong_wind,html,26047
5,SRC_009,Outdoor Emergency Plan,US National Park Service (NPS),remote_emergency,html,76363
6,SRC_010,Ten Essentials,US National Park Service (NPS),remote_emergency,html,49027
7,SRC_014,Hypothermia,NHS (England),hypothermia,html,32978
8,SRC_015,Heat exhaustion and heatstroke,NHS (England),heat_illness,html,33587
9,SRC_016,Cuts and grazes,NHS (England),general_first_aid,html,33991


In [8]:
def extract_categories(value):

    if pd.isna(value):
        return []

    text = str(value)

    for separator in [
        "|",
        ";",
        ","
    ]:
        text = text.replace(
            separator,
            "|"
        )

    return [
        item.strip()
        for item in text.split("|")
        if item.strip()
    ]


category_counts = {
    category: 0
    for category in RAG_CATEGORIES
}

unknown_categories = set()

for value in offline_catalog[
    "categories"
]:

    for category in extract_categories(
        value
    ):

        if category in category_counts:

            category_counts[
                category
            ] += 1

        else:

            unknown_categories.add(
                category
            )


category_coverage = pd.DataFrame({
    "category":
        list(
            category_counts.keys()
        ),

    "offline_source_count":
        list(
            category_counts.values()
        )
})

category_coverage[
    "has_offline_coverage"
] = (
    category_coverage[
        "offline_source_count"
    ] > 0
)

print("OFFLINE CATEGORY COVERAGE")
print("=" * 60)

display(
    category_coverage
)

print(
    "Unknown category values:",
    sorted(
        unknown_categories
    )
)

print(
    "All required categories covered:",
    bool(
        category_coverage[
            "has_offline_coverage"
        ].all()
    )
)

OFFLINE CATEGORY COVERAGE


,category,offline_source_count,has_offline_coverage
0,general_first_aid,2,True
1,hypothermia,2,True
2,heat_illness,2,True
3,heavy_rain,2,True
4,strong_wind,2,True
5,water_safety,3,True
6,remote_emergency,3,True


Unknown category values: []
All required categories covered: True


## Offline Knowledge Text Extraction



In [9]:
import importlib.util

required_packages = {
    "BeautifulSoup": "bs4",
    "PyPDF": "pypdf"
}

print("TEXT EXTRACTION DEPENDENCIES")
print("=" * 60)

dependency_status = {}

for name, module in required_packages.items():

    installed = (
        importlib.util.find_spec(
            module
        ) is not None
    )

    dependency_status[name] = installed

    print(
        f"{name}: {installed}"
    )

TEXT EXTRACTION DEPENDENCIES
BeautifulSoup: True
PyPDF: True


In [10]:
from bs4 import BeautifulSoup
import re


def clean_whitespace(text):

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n[ \t]+",
        "\n",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


def extract_html_text(path):

    html = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    # Remove elements that normally contain
    # non-knowledge webpage content.
    for element in soup(
        [
            "script",
            "style",
            "noscript",
            "svg",
            "form"
        ]
    ):
        element.decompose()

    # Prefer the semantic main/article content
    # when the webpage provides it.
    content = (
        soup.find("main")
        or soup.find("article")
        or soup.body
        or soup
    )

    text = content.get_text(
        separator="\n",
        strip=True
    )

    return clean_whitespace(
        text
    )

In [11]:
from pypdf import PdfReader


def extract_pdf_text(path):

    reader = PdfReader(
        str(path)
    )

    page_texts = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        text = (
            page.extract_text()
            or ""
        )

        text = clean_whitespace(
            text
        )

        if text:
            page_texts.append(
                text
            )

    combined_text = "\n\n".join(
        page_texts
    )

    return (
        clean_whitespace(
            combined_text
        ),
        len(reader.pages)
    )

In [12]:
extraction_records = []

for _, row in offline_catalog.iterrows():

    path = row[
        "resolved_local_path"
    ]

    file_type = str(
        row["file_type"]
    ).lower().strip()

    status = "success"
    error = None
    page_count = None
    text = ""

    try:

        if file_type == "pdf":

            text, page_count = (
                extract_pdf_text(
                    path
                )
            )

        elif file_type in [
            "html",
            "htm"
        ]:

            text = extract_html_text(
                path
            )

        else:

            status = (
                "unsupported_file_type"
            )

    except Exception as exc:

        status = "error"
        error = str(exc)

    extraction_records.append({
        "source_id":
            row["source_id"],

        "title":
            row["title"],

        "organization":
            row["organization"],

        "categories":
            row["categories"],

        "file_type":
            file_type,

        "page_count":
            page_count,

        "extraction_status":
            status,

        "character_count":
            len(text),

        "word_count":
            len(text.split()),

        "error":
            error,

        "extracted_text":
            text
    })


extracted_documents = pd.DataFrame(
    extraction_records
)

print("TEXT EXTRACTION")
print("=" * 60)

display(
    extracted_documents[
        [
            "source_id",
            "title",
            "file_type",
            "page_count",
            "extraction_status",
            "character_count",
            "word_count",
            "error"
        ]
    ]
)

TEXT EXTRACTION


,source_id,title,file_type,page_count,extraction_status,character_count,word_count,error
0,SRC_001,"IFRC International First Aid, Resuscitation an...",pdf,594.0,success,1527421,227058,None
1,SRC_005,During a Flood,html,NaN,success,2638,410,None
2,SRC_006,Lightning Safety,html,NaN,success,1876,267,None
3,SRC_007,During a High Wind Event,html,NaN,success,2930,484,None
4,SRC_008,Before a High Wind Event,html,NaN,success,2022,314,None
5,SRC_009,Outdoor Emergency Plan,html,NaN,success,10199,1845,None
6,SRC_010,Ten Essentials,html,NaN,success,6230,1043,None
7,SRC_014,Hypothermia,html,NaN,success,2338,425,None
8,SRC_015,Heat exhaustion and heatstroke,html,NaN,success,3823,663,None
9,SRC_016,Cuts and grazes,html,NaN,success,4280,839,None


In [13]:
print("EXTRACTION VALIDATION")
print("=" * 60)

print(
    "Documents attempted:",
    len(extracted_documents)
)

print(
    "Successful:",
    (
        extracted_documents[
            "extraction_status"
        ] == "success"
    ).sum()
)

print(
    "Failed:",
    (
        extracted_documents[
            "extraction_status"
        ] != "success"
    ).sum()
)

print(
    "Empty documents:",
    (
        extracted_documents[
            "character_count"
        ] == 0
    ).sum()
)

print(
    "Documents under 100 words:",
    (
        extracted_documents[
            "word_count"
        ] < 100
    ).sum()
)

print(
    "Total extracted words:",
    extracted_documents[
        "word_count"
    ].sum()
)

EXTRACTION VALIDATION
Documents attempted: 11
Successful: 11
Failed: 0
Empty documents: 0
Documents under 100 words: 0
Total extracted words: 234465


In [14]:
for _, row in extracted_documents.iterrows():

    print("\n")
    print("=" * 80)

    print(
        row["source_id"],
        "-",
        row["title"]
    )

    print(
        "Organization:",
        row["organization"]
    )

    print(
        "Words:",
        row["word_count"]
    )

    print("-" * 80)

    preview = row[
        "extracted_text"
    ][:1500]

    print(preview)

    print("\n[PREVIEW END]")



SRC_001 - IFRC International First Aid, Resuscitation and Education Guidelines 2025
Organization: IFRC
Words: 227058
--------------------------------------------------------------------------------
INTERNATIONAL FIRST AID, 
RESUSCITATION AND EDUCATION 
GUIDELINES 2025

Contents 
TOPIC INDEX 4 
INTRODUCTION 11 
CONTEXTUALIZING FIRST AID AND FIRST AID EDUCATION 27 
FIRST AID EDUCATION 71 
FIRST AID 111 
GENERAL APPROACH 113 
RESUSCITATION 151 
BREATHING PROBLEMS 231 
TRAUMA 255 
MEDICAL CONDITIONS 357 
ENVIRONMENTAL 445 
MENTAL DISTRESS 485 
GLOSSARY AND REFERENCES 523 

Audience: First aid programme designers, programme managers, education and scienti ﬁc 
committees, trainers 

Red Cross Red Crescent Networks 
Coordinated by IFRC Global First Aid Reference Centre 

© International Federation of Red Cross and Red Crescent Societies, Geneva, 2025 
Copies of all or part of this study may be made for non-commercial use, providing the source is 
acknowledged. The IFRC would appreciate rece

## Knowledge Cleaning and Chunking Strategy








In [15]:
def clean_extracted_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize line endings
    text = text.replace(
        "\r\n",
        "\n"
    ).replace(
        "\r",
        "\n"
    )

    # Remove excessive spaces
    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    # Remove spaces around newlines
    text = re.sub(
        r" *\n *",
        "\n",
        text
    )

    # Limit excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


extracted_documents[
    "cleaned_text"
] = extracted_documents[
    "extracted_text"
].apply(
    clean_extracted_text
)

extracted_documents[
    "cleaned_word_count"
] = extracted_documents[
    "cleaned_text"
].apply(
    lambda text: len(
        text.split()
    )
)

print("TEXT CLEANING")
print("=" * 60)

display(
    extracted_documents[
        [
            "source_id",
            "word_count",
            "cleaned_word_count"
        ]
    ]
)

TEXT CLEANING


,source_id,word_count,cleaned_word_count
0,SRC_001,227058,227058
1,SRC_005,410,410
2,SRC_006,267,267
3,SRC_007,484,484
4,SRC_008,314,314
5,SRC_009,1845,1845
6,SRC_010,1043,1043
7,SRC_014,425,425
8,SRC_015,663,663
9,SRC_016,839,839


In [16]:
NOISE_PATTERNS = [
    "accept cookies",
    "cookie policy",
    "skip to main content",
    "privacy policy",
    "accessibility",
    "javascript",
    "menu",
    "search this site",
    "back to top",
    "sign in"
]

noise_records = []

for _, row in extracted_documents.iterrows():

    text_lower = (
        row["cleaned_text"]
        .lower()
    )

    found_patterns = [
        pattern
        for pattern in NOISE_PATTERNS
        if pattern in text_lower
    ]

    noise_records.append({
        "source_id":
            row["source_id"],

        "title":
            row["title"],

        "noise_pattern_count":
            len(found_patterns),

        "patterns":
            "; ".join(
                found_patterns
            )
    })


noise_audit = pd.DataFrame(
    noise_records
)

print("WEB/PAGE NOISE AUDIT")
print("=" * 60)

display(
    noise_audit
)

WEB/PAGE NOISE AUDIT


,source_id,title,noise_pattern_count,patterns
0,SRC_001,"IFRC International First Aid, Resuscitation an...",1,accessibility
1,SRC_005,During a Flood,1,privacy policy
2,SRC_006,Lightning Safety,1,privacy policy
3,SRC_007,During a High Wind Event,1,privacy policy
4,SRC_008,Before a High Wind Event,1,privacy policy
5,SRC_009,Outdoor Emergency Plan,3,privacy policy; accessibility; menu
6,SRC_010,Ten Essentials,3,privacy policy; accessibility; menu
7,SRC_014,Hypothermia,0,
8,SRC_015,Heat exhaustion and heatstroke,0,
9,SRC_016,Cuts and grazes,0,


In [17]:
CHUNK_SIZE_WORDS = 180
CHUNK_OVERLAP_WORDS = 40


def chunk_text(
    text,
    chunk_size=CHUNK_SIZE_WORDS,
    overlap=CHUNK_OVERLAP_WORDS
):

    words = text.split()

    if not words:
        return []

    chunks = []

    start = 0

    while start < len(words):

        end = min(
            start + chunk_size,
            len(words)
        )

        chunk = " ".join(
            words[start:end]
        ).strip()

        if chunk:
            chunks.append(
                chunk
            )

        if end >= len(words):
            break

        start = (
            end - overlap
        )

    return chunks

In [18]:
catalog_lookup = (
    offline_catalog
    .set_index("source_id")
)

chunk_records = []

global_chunk_number = 1

for _, document in extracted_documents.iterrows():

    source_id = document[
        "source_id"
    ]

    source_metadata = (
        catalog_lookup.loc[
            source_id
        ]
    )

    chunks = chunk_text(
        document[
            "cleaned_text"
        ]
    )

    for local_index, chunk in enumerate(
        chunks,
        start=1
    ):

        chunk_records.append({
            "chunk_id":
                f"CHK_{global_chunk_number:05d}",

            "source_id":
                source_id,

            "source_chunk_index":
                local_index,

            "title":
                source_metadata[
                    "title"
                ],

            "organization":
                source_metadata[
                    "organization"
                ],

            "categories":
                source_metadata[
                    "categories"
                ],

            "source_url":
                source_metadata[
                    "source_url"
                ],

            "word_count":
                len(
                    chunk.split()
                ),

            "content":
                chunk
        })

        global_chunk_number += 1


knowledge_chunks = pd.DataFrame(
    chunk_records
)

print("KNOWLEDGE CHUNKING")
print("=" * 60)

print(
    "Total chunks:",
    len(knowledge_chunks)
)

print(
    "Duplicate chunk IDs:",
    knowledge_chunks[
        "chunk_id"
    ].duplicated().sum()
)

print(
    "Empty chunks:",
    (
        knowledge_chunks[
            "content"
        ].str.strip() == ""
    ).sum()
)

print(
    "Mean words per chunk:",
    round(
        knowledge_chunks[
            "word_count"
        ].mean(),
        2
    )
)

print(
    "Maximum words:",
    knowledge_chunks[
        "word_count"
    ].max()
)

print(
    "Minimum words:",
    knowledge_chunks[
        "word_count"
    ].min()
)

KNOWLEDGE CHUNKING
Total chunks: 1676
Duplicate chunk IDs: 0
Empty chunks: 0
Mean words per chunk: 179.63
Maximum words: 180
Minimum words: 63


In [19]:
chunk_distribution = (
    knowledge_chunks
    .groupby(
        [
            "source_id",
            "title"
        ]
    )
    .size()
    .reset_index(
        name="chunk_count"
    )
    .sort_values(
        "chunk_count",
        ascending=False
    )
)

chunk_distribution[
    "corpus_percent"
] = (
    chunk_distribution[
        "chunk_count"
    ]
    /
    len(
        knowledge_chunks
    )
    * 100
)

print("CHUNK DISTRIBUTION")
print("=" * 60)

display(
    chunk_distribution
)

CHUNK DISTRIBUTION


,source_id,title,chunk_count,corpus_percent
0,SRC_001,"IFRC International First Aid, Resuscitation an...",1622,96.778043
5,SRC_009,Outdoor Emergency Plan,13,0.775656
6,SRC_010,Ten Essentials,8,0.477327
10,SRC_017,Emergency Disinfection of Drinking Water,8,0.477327
9,SRC_016,Cuts and grazes,6,0.357995
8,SRC_015,Heat exhaustion and heatstroke,5,0.298329
3,SRC_007,During a High Wind Event,4,0.238663
1,SRC_005,During a Flood,3,0.178998
7,SRC_014,Hypothermia,3,0.178998
2,SRC_006,Lightning Safety,2,0.119332


In [20]:
required_chunk_columns = [
    "chunk_id",
    "source_id",
    "source_chunk_index",
    "title",
    "organization",
    "categories",
    "source_url",
    "word_count",
    "content"
]

missing_chunk_columns = [
    column
    for column in required_chunk_columns
    if column not in knowledge_chunks.columns
]

unknown_source_ids = (
    set(
        knowledge_chunks[
            "source_id"
        ]
    )
    -
    set(
        offline_catalog[
            "source_id"
        ]
    )
)

missing_provenance = (
    knowledge_chunks[
        [
            "source_id",
            "title",
            "organization",
            "source_url"
        ]
    ]
    .isna()
    .sum()
    .sum()
)

print("CHUNK VALIDATION")
print("=" * 60)

print(
    "Missing required columns:",
    missing_chunk_columns
)

print(
    "Unknown source IDs:",
    unknown_source_ids
)

print(
    "Missing provenance values:",
    missing_provenance
)

print(
    "Duplicate chunk IDs:",
    knowledge_chunks[
        "chunk_id"
    ].duplicated().sum()
)

print(
    "Valid:",
    (
        len(missing_chunk_columns) == 0
        and len(unknown_source_ids) == 0
        and missing_provenance == 0
        and knowledge_chunks[
            "chunk_id"
        ].duplicated().sum() == 0
    )
)

CHUNK VALIDATION
Missing required columns: []
Unknown source IDs: set()
Missing provenance values: 0
Duplicate chunk IDs: 0
Valid: True


In [21]:
preview_sources = [
    "SRC_001",
    "SRC_005",
    "SRC_014",
    "SRC_017"
]

for source_id in preview_sources:

    sample = (
        knowledge_chunks[
            knowledge_chunks[
                "source_id"
            ] == source_id
        ]
        .head(2)
    )

    print("\n")
    print("=" * 80)
    print(source_id)

    for _, row in sample.iterrows():

        print(
            "\n",
            row["chunk_id"],
            "|",
            row["organization"]
        )

        print("-" * 80)

        print(
            row["content"]
        )



SRC_001

 CHK_00001 | IFRC
--------------------------------------------------------------------------------
INTERNATIONAL FIRST AID, RESUSCITATION AND EDUCATION GUIDELINES 2025 Contents TOPIC INDEX 4 INTRODUCTION 11 CONTEXTUALIZING FIRST AID AND FIRST AID EDUCATION 27 FIRST AID EDUCATION 71 FIRST AID 111 GENERAL APPROACH 113 RESUSCITATION 151 BREATHING PROBLEMS 231 TRAUMA 255 MEDICAL CONDITIONS 357 ENVIRONMENTAL 445 MENTAL DISTRESS 485 GLOSSARY AND REFERENCES 523 Audience: First aid programme designers, programme managers, education and scienti ﬁc committees, trainers Red Cross Red Crescent Networks Coordinated by IFRC Global First Aid Reference Centre © International Federation of Red Cross and Red Crescent Societies, Geneva, 2025 Copies of all or part of this study may be made for non-commercial use, providing the source is acknowledged. The IFRC would appreciate receiving details of its use. Requests for commercial reproduction should be directed to the IFRC at secretariat@ifrc.or

## Offline SQLite Retrieval Layer



In [22]:
CHUNKS_FILE = (
    PROCESSED_DIR
    / "knowledge_chunks.csv"
)

knowledge_chunks.to_csv(
    CHUNKS_FILE,
    index=False
)

print("PROCESSED CORPUS")
print("=" * 60)

print(
    "Saved:",
    CHUNKS_FILE.exists()
)

print(
    "Rows:",
    len(knowledge_chunks)
)

print(
    "Path:",
    CHUNKS_FILE
)

PROCESSED CORPUS
Saved: True
Rows: 1676
Path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/processed/knowledge_chunks.csv


In [23]:
DATABASE_FILE = (
    DATABASE_DIR
    / "offline_hiking_knowledge.db"
)

if DATABASE_FILE.exists():
    DATABASE_FILE.unlink()

connection = sqlite3.connect(
    DATABASE_FILE
)

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE documents (
    source_id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    organization TEXT NOT NULL,
    categories TEXT NOT NULL,
    source_url TEXT NOT NULL,
    file_type TEXT,
    local_path TEXT
)
""")

cursor.execute("""
CREATE TABLE knowledge_chunks (
    chunk_id TEXT PRIMARY KEY,
    source_id TEXT NOT NULL,
    source_chunk_index INTEGER NOT NULL,
    categories TEXT NOT NULL,
    word_count INTEGER NOT NULL,
    content TEXT NOT NULL,
    FOREIGN KEY (source_id)
        REFERENCES documents(source_id)
)
""")

connection.commit()

print("SQLITE DATABASE")
print("=" * 60)

print(
    "Created:",
    DATABASE_FILE.exists()
)

print(
    "Path:",
    DATABASE_FILE
)

SQLITE DATABASE
Created: True
Path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db


In [24]:
document_rows = []

for _, row in offline_catalog.iterrows():

    document_rows.append((
        str(row["source_id"]),
        str(row["title"]),
        str(row["organization"]),
        str(row["categories"]),
        str(row["source_url"]),
        str(row["file_type"]),
        str(row["local_path"])
    ))

cursor.executemany("""
INSERT INTO documents (
    source_id,
    title,
    organization,
    categories,
    source_url,
    file_type,
    local_path
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", document_rows)

connection.commit()

document_count = cursor.execute(
    "SELECT COUNT(*) FROM documents"
).fetchone()[0]

print("DOCUMENT INSERTION")
print("=" * 60)

print(
    "Documents inserted:",
    document_count
)


DOCUMENT INSERTION
Documents inserted: 11


In [25]:
chunk_rows = []

for _, row in knowledge_chunks.iterrows():

    chunk_rows.append((
        str(row["chunk_id"]),
        str(row["source_id"]),
        int(row["source_chunk_index"]),
        str(row["categories"]),
        int(row["word_count"]),
        str(row["content"])
    ))

cursor.executemany("""
INSERT INTO knowledge_chunks (
    chunk_id,
    source_id,
    source_chunk_index,
    categories,
    word_count,
    content
)
VALUES (?, ?, ?, ?, ?, ?)
""", chunk_rows)

connection.commit()

chunk_count = cursor.execute(
    "SELECT COUNT(*) FROM knowledge_chunks"
).fetchone()[0]

print("CHUNK INSERTION")
print("=" * 60)

print(
    "Chunks inserted:",
    chunk_count
)

CHUNK INSERTION
Chunks inserted: 1676


In [26]:
cursor.execute("""
CREATE VIRTUAL TABLE knowledge_chunks_fts
USING fts5(
    chunk_id UNINDEXED,
    source_id UNINDEXED,
    categories UNINDEXED,
    content
)
""")

cursor.execute("""
INSERT INTO knowledge_chunks_fts (
    chunk_id,
    source_id,
    categories,
    content
)
SELECT
    chunk_id,
    source_id,
    categories,
    content
FROM knowledge_chunks
""")

connection.commit()

fts_count = cursor.execute(
    "SELECT COUNT(*) FROM knowledge_chunks_fts"
).fetchone()[0]

print("FTS5 INDEX")
print("=" * 60)

print(
    "Indexed chunks:",
    fts_count
)

FTS5 INDEX
Indexed chunks: 1676


In [27]:
database_validation = {
    "documents":
        cursor.execute(
            "SELECT COUNT(*) FROM documents"
        ).fetchone()[0],

    "knowledge_chunks":
        cursor.execute(
            "SELECT COUNT(*) FROM knowledge_chunks"
        ).fetchone()[0],

    "fts_chunks":
        cursor.execute(
            "SELECT COUNT(*) FROM knowledge_chunks_fts"
        ).fetchone()[0]
}

print("DATABASE VALIDATION")
print("=" * 60)

for key, value in database_validation.items():
    print(
        f"{key}: {value}"
    )

print()

database_valid = (
    database_validation[
        "documents"
    ] == 11

    and

    database_validation[
        "knowledge_chunks"
    ] == 1676

    and

    database_validation[
        "fts_chunks"
    ] == 1676
)

print(
    "Database valid:",
    database_valid
)

print(
    "Database size:",
    f"{DATABASE_FILE.stat().st_size / (1024 ** 2):.2f} MB"
)

DATABASE VALIDATION
documents: 11
knowledge_chunks: 1676
fts_chunks: 1676

Database valid: True
Database size: 6.27 MB


In [28]:
def prepare_fts_query(question):

    terms = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    stop_words = {
        "a", "an", "the", "and", "or",
        "is", "are", "was", "were",
        "i", "me", "my", "we", "our",
        "to", "of", "for", "in", "on",
        "at", "with", "if", "do", "does",
        "what", "should", "how", "can"
    }

    terms = [
        term
        for term in terms
        if term not in stop_words
        and len(term) > 1
    ]

    if not terms:
        return None

    return " OR ".join(
        f'"{term}"'
        for term in terms
    )


def retrieve_chunks(
    question,
    category=None,
    limit=5
):

    fts_query = prepare_fts_query(
        question
    )

    if not fts_query:
        return pd.DataFrame()

    parameters = [
        fts_query
    ]

    category_condition = ""

    if category is not None:

        category_condition = """
        AND kc.categories LIKE ?
        """

        parameters.append(
            f"%{category}%"
        )

    parameters.append(
        max(
            limit * 5,
            20
        )
    )

    sql = f"""
    SELECT
        kc.chunk_id,
        kc.source_id,
        d.title,
        d.organization,
        kc.categories,
        d.source_url,
        kc.content,
        bm25(knowledge_chunks_fts) AS rank
    FROM knowledge_chunks_fts
    JOIN knowledge_chunks kc
        ON kc.chunk_id =
           knowledge_chunks_fts.chunk_id
    JOIN documents d
        ON d.source_id = kc.source_id
    WHERE knowledge_chunks_fts
          MATCH ?
    {category_condition}
    ORDER BY rank
    LIMIT ?
    """

    results = pd.read_sql_query(
        sql,
        connection,
        params=parameters
    )

    if results.empty:
        return results

    # Source diversification:
    # prevent one large source from occupying
    # every returned position.
    selected_rows = []
    source_counts = {}

    for _, row in results.iterrows():

        source_id = row[
            "source_id"
        ]

        count = source_counts.get(
            source_id,
            0
        )

        if count >= 2:
            continue

        selected_rows.append(
            row
        )

        source_counts[
            source_id
        ] = count + 1

        if len(
            selected_rows
        ) >= limit:
            break

    return pd.DataFrame(
        selected_rows
    ).reset_index(
        drop=True
    )

In [29]:
test_question = (
    "What should I do if a hiker "
    "shows signs of hypothermia?"
)

results = retrieve_chunks(
    question=test_question,
    category="hypothermia",
    limit=5
)

print("OFFLINE RETRIEVAL TEST")
print("=" * 60)

print(
    "Question:",
    test_question
)

print(
    "Results:",
    len(results)
)

display(
    results[
        [
            "chunk_id",
            "source_id",
            "organization",
            "rank"
        ]
    ]
)

for _, row in results.iterrows():

    print("\n")
    print(
        row["chunk_id"],
        "|",
        row["organization"]
    )

    print("-" * 70)

    print(
        row["content"][:1000]
    )

OFFLINE RETRIEVAL TEST
Question: What should I do if a hiker shows signs of hypothermia?
Results: 4


,chunk_id,source_id,organization,rank
0,CHK_01246,SRC_001,IFRC,-8.259663
1,CHK_01247,SRC_001,IFRC,-7.089062
2,CHK_01655,SRC_014,NHS (England),-7.029797
3,CHK_01657,SRC_014,NHS (England),-6.920280




CHK_01246 | IFRC
----------------------------------------------------------------------
muscles further. ● Do not apply warm stones directly to the skin as this may burn the person. ● If the person becomes unresponsive, open their airway and check for breathing. See Unresponsiveness. ● In a limited-resource setting, create a heat source (such as a campﬁre) to warm the person and build a shelter to protect them from the cold. Ensure there is a layer, such as a blanket, between the person and the ground. Access help For all cases of hypothermia, the ﬁrst aid provider should access emergency medical care immediately. Recovery Most healthy people with mild to moderate hypothermia will recover fully. However, if signs and symptoms continue or additional ones appear, seek medical care. Education considerations Context considerations ● Learners in or visiting cold climates may bene ﬁt from learning how to prevent hypothermia and frostbite (including winter conditions in any setting, and spe